# 6단계: 추천 평가

5단계 파이프라인의 설정(텍스트 A/B, 모드, 상한, 후보 수, 가중치)을 정답 세트 기준으로 비교한다.

정답 세트 만드는 방법
1. 여러 설정의 상위 5개를 합쳐 질의별 **판정 풀**을 만든다 (풀링 평가).
2. 풀의 각 (질의, 메뉴)에 적합도를 매긴다. 2 적합, 1 부분 적합, 0 부적합.
3. 판정은 `판정출처`와 `검토상태`를 갖는다. 현재 판정은 3단계 라벨링과 같은 방식으로 Claude가 **메뉴명·업체명·분류만 보고** 매긴 모델 추정이며
   (저장된 속성 라벨은 참조하지 않아 평가가 라벨에 순환하지 않는다) 전부 `검토대기` 상태다.
4. 사람이 `data/processed/evaluation/judgments.csv`의 적합도를 고치고 `검토상태`를 `승인`으로 바꾸면, 승인된 판정만으로 다시 계산한다 (10절).

주의할 점
- 아래 지표는 **모델 추정 정답 기준**이며 최종 성능이 아니다. 승인 전에는 설정 간 상대 비교의 참고치로만 쓴다.
- 풀에 없는 항목은 미판정이며 0으로 취급한다. 풀에 들어간 설정이 유리하므로 미판정 비율을 반드시 함께 본다.
- 정답 세트가 22개 질의 중 18개, 296건뿐이라 작은 차이는 의미가 없다.
- 입력: `data/processed/recommendation/run_config.json`(질의), `data/embeddings/`(A, B) / 출력: `data/processed/evaluation/`

## 1. 모듈 로드

In [1]:
import json
import sys
import time
from collections import Counter
from dataclasses import asdict
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "src").is_dir() and (p / "data").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Menu_recommend 저장소 안에서 실행해 주세요")
sys.path.insert(0, str(PROJECT_ROOT))

from src.embedding import DEFAULT_SPEC, E5Embedder, describe_environment
from src.preprocessing import parse_query
from src.ranking import RankingConfig
from src.recommendation import EMBEDDING_ONLY, FILTER_ONLY, FULL, PipelineConfig, Recommender
from src.recommendation.evaluation import (
    APPROVED, PENDING, UNJUDGED, build_pool, evaluate_configs, evaluate_result, judgment_map, load_judgments,
    merge_pool, save_judgments,
)
from src.retrieval import load_index

REC_DIR = PROJECT_ROOT / "data" / "processed" / "recommendation"
OUT_DIR = PROJECT_ROOT / "data" / "processed" / "evaluation"
OUT_DIR.mkdir(parents=True, exist_ok=True)
JUDGMENTS_PATH = OUT_DIR / "judgments.csv"
K = 5

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 300)
describe_environment()

{'platform': 'macOS-26.6.2-arm64-arm-64bit',
 'processor': 'arm',
 'cpu_count': 12,
 'total_memory_gb': 32.0,
 'torch_version': '2.14.0',
 'cuda_available': False,
 'mps_available': True,
 'selected_device': 'mps'}

## 2. 평가 질의

5단계 비교 실험의 22개 질의 중 정답을 정의할 수 있는 18개를 쓴다.
모순 질의와 빈 입력은 0건 반환이 정답이고, "매운 것도 괜찮아"는 조건이 없어 무엇이 적합한지 정할 수 없어 제외한다.

In [2]:
run5 = json.load(open(REC_DIR / "run_config.json", encoding="utf-8"))
EXCLUDED = {"": "빈 입력, 0건 반환이 정답", "맵지 않은 매운 음식": "모순, 0건 반환이 정답",
            "국물 없는 국물 요리": "모순, 0건 반환이 정답", "매운 것도 괜찮아": "조건 없음, 적합 기준을 정할 수 없음"}
QUERY_KIND = {q["질의"]: q["유형"] for q in run5["질의"]}
QUERIES = [q["질의"] for q in run5["질의"] if q["질의"] not in EXCLUDED]
print(f"평가 질의 {len(QUERIES)}개, 제외 {len(EXCLUDED)}개")
pd.DataFrame([{"질의": q or "(빈 입력)", "제외 사유": r} for q, r in EXCLUDED.items()])

평가 질의 18개, 제외 4개


,질의,제외 사유
0,(빈 입력),"빈 입력, 0건 반환이 정답"
1,맵지 않은 매운 음식,"모순, 0건 반환이 정답"
2,국물 없는 국물 요리,"모순, 0건 반환이 정답"
3,매운 것도 괜찮아,"조건 없음, 적합 기준을 정할 수 없음"


## 3. 추천기 로드 (텍스트 A, B)

In [3]:
t0 = time.time()
embedder = E5Embedder(spec=DEFAULT_SPEC)
encode_query = lambda text: embedder.encode_queries([text], show_progress=False)[0]
RECS = {}
for variant in ("A", "B"):
    index, ref = load_index(variant)
    RECS[variant] = Recommender(index, encode_query, ref)
    print(f"텍스트 {variant}: {ref['name']} ({ref['num_vectors']}건)")
print(f"로드 {time.time() - t0:.1f}초, device={embedder.device}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

/Users/jack/project/Menu-recommend-algorithmn/src/embedding/embedder.py:123: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  loaded_dim = self.model.get_sentence_embedding_dimension()


텍스트 A: multilingual-e5-base_d1287505_textA_v1_eb332ded (5219건)
텍스트 B: multilingual-e5-base_d1287505_textB_v1_f87e895f (5219건)
로드 7.3초, device=mps


## 4. 판정 풀과 판정 시트

풀에 넣는 설정: A·B 각각 3개 모드(임베딩만, 조건적용, 전체) + B의 상한 0/1, 감점 0.02, 후보 100 고정, 후보 400 고정.
기존 시트가 있으면 판정과 검토상태를 그대로 두고 새로 들어온 항목만 미판정으로 덧붙인다.

In [4]:
MODES = {"임베딩만": EMBEDDING_ONLY, "조건적용": FILTER_ONLY, "조건+재랭킹+중복제어": FULL}
VARIANTS = {
    "상한0": PipelineConfig(ranking=RankingConfig(group_cap=0)),
    "상한1": PipelineConfig(ranking=RankingConfig(group_cap=1)),
    "감점0.02": PipelineConfig(ranking=RankingConfig(group_penalty=0.02)),
    "후보100 고정": PipelineConfig(candidate_k=100, preference_widen_k=100),
    "후보400 고정": PipelineConfig(candidate_k=400, preference_widen_k=400),
}
pool = build_pool([RECS["A"], RECS["B"]], QUERIES, list(MODES.values()), k=K)
seen = {(p["질의"], p["라벨링단위ID"]) for p in pool}
pool += [p for p in build_pool([RECS["B"]], QUERIES, list(VARIANTS.values()), k=K)
         if (p["질의"], p["라벨링단위ID"]) not in seen]

existing = load_judgments(JUDGMENTS_PATH)
criteria = {r["질의"]: r["판정기준"] for r in existing if r.get("판정기준")}
rows = merge_pool(existing, pool, criteria)
save_judgments(JUDGMENTS_PATH, rows)
status = Counter(r["검토상태"] for r in rows)
print(f"풀 {len(pool)}건, 시트 {len(rows)}행 (기존 {len(existing)}, 새로 추가 {len(rows) - len(existing)})")
print("검토상태:", dict(status))
print("판정출처:", dict(Counter(r["판정출처"] for r in rows if r["판정출처"])))

풀 296건, 시트 296행 (기존 296, 새로 추가 0)
검토상태: {'검토대기': 296}
판정출처: {'모델추정 (Claude, 메뉴명·업체명·분류만 참조)': 296}


In [5]:
# 질의별 판정 기준
pd.DataFrame([{"질의": q, "판정기준": criteria.get(q, "-")} for q in QUERIES])

,질의,판정기준
0,비 오는 날 얼큰한 국물 먹고 싶어,"2: 매콤한 뜨거운 국물 요리 / 1: 국물 요리지만 안 매움, 또는 매운 비국물 / 0: 그 외"
1,맵지 않고 따뜻한 음식,2: 안 맵고 따뜻하게 먹는 음식 / 1: 안 맵지만 차갑거나 상온 / 0: 매운 음식
2,차갑고 가볍게 먹을 메뉴,"2: 차갑고 가벼운 음식(냉면·냉국·샐러드·아이스류) / 1: 카페 냉장 샌드위치, 가볍지만 따뜻함, 차갑지만 무거움 / 0: 뜨겁고 무거움"
3,바삭하고 기름진 음식,2: 튀김·치킨·튀긴 패티 / 1: 기름지지만 바삭하지 않음(피자 등) / 0: 국·면·죽
4,든든한 밥 한 끼,"2: 밥 중심 한 끼(덮밥·볶음밥·비빔밥·국밥) / 1: 밥 아닌 든든한 식사, 흰밥·김밥 / 0: 간식"
5,국물 없는 매운 음식,"2: 매운 비국물 음식 / 1: 약간 매운 비국물, 안 매운 비국물 / 0: 국물 요리"
6,상큼하고 시원한 음식,"2: 차갑고 새콤한 음식(냉면·비빔국수·쫄면) / 1: 차갑지만 상큼하지 않음, 이름만 상큼 / 0: 뜨거운 음식"
7,포만감 있는 저녁밥,"2: 든든한 밥 식사 / 1: 밥 아닌 식사, 흰밥 / 0: 간식·가벼운 음식"
8,빠르게 먹을 수 있는 간식,2: 간단히 먹는 간식(샌드위치·주먹밥·핫도그·꼬치) / 1: 버거·피자·라면·죽 / 0: 정찬·면 식사
9,따뜻한 국이나 찌개,2: 국·찌개·탕·전골 / 1: 국물이 자작한 요리 / 0: 비국물


In [6]:
# 질의별 적합도 분포
sheet = pd.DataFrame(rows)
sheet["적합도"] = pd.to_numeric(sheet["적합도"], errors="coerce")
dist = sheet.groupby("질의")["적합도"].agg(
    판정수="count", 적합=lambda s: int((s == 2).sum()), 부분=lambda s: int((s == 1).sum()),
    부적합=lambda s: int((s == 0).sum()), 미판정=lambda s: int(s.isna().sum()))
dist.loc[QUERIES]

,판정수,적합,부분,부적합,미판정
질의,,,,,
비 오는 날 얼큰한 국물 먹고 싶어,19,9,9,1,0
맵지 않고 따뜻한 음식,17,12,4,1,0
차갑고 가볍게 먹을 메뉴,19,2,6,11,0
바삭하고 기름진 음식,18,9,7,2,0
든든한 밥 한 끼,18,5,12,1,0
국물 없는 매운 음식,21,11,3,7,0
상큼하고 시원한 음식,15,4,3,8,0
포만감 있는 저녁밥,20,13,7,0,0
빠르게 먹을 수 있는 간식,17,6,9,2,0


## 5. 지표

- P@5: 상위 5개 중 적합(2) 비율
- nDCG@5: 적합도를 이득(2^g − 1)으로 한 정규화 누적 이득, 이상적 순서는 그 질의의 판정 전체 기준
- MRR: 첫 적합(2) 항목 순위의 역수 평균
- 미판정비율: 반환 항목 중 판정되지 않은 비율. 미판정은 0으로 계산되므로 이 값이 큰 설정은 과소평가된다

In [7]:
jmap = judgment_map(rows)
print(f"판정 {len(jmap)}건 (모델 추정 포함)")
CONFIGS = {**MODES, **VARIANTS,
           "선호가중치0": PipelineConfig(ranking=RankingConfig(similarity_weight=1.0, preference_weight=0.0))}
t0 = time.time()
eval_rows = evaluate_configs(RECS, QUERIES, CONFIGS, jmap, k=K)
eval_df = pd.DataFrame(eval_rows).round(4)
print(f"{len(eval_rows)}개 조합 평가, {time.time() - t0:.1f}초")
eval_df.pivot(index="설정", columns="텍스트구성", values=[f"P@{K}", f"nDCG@{K}", "MRR", "미판정비율"]).loc[list(CONFIGS)]

판정 296건 (모델 추정 포함)


18개 조합 평가, 0.4초


P@5          nDCG@5             MRR           미판정비율        
텍스트구성             A       B       A       B       A       B       A       B
설정                                                                         
임베딩만         0.3111  0.3111  0.4040  0.4321  0.3833  0.4491  0.0000  0.0000
조건적용         0.5000  0.4444  0.5885  0.5763  0.5185  0.5324  0.0000  0.0000
조건+재랭킹+중복제어  0.7556  0.7333  0.8363  0.8329  0.8241  0.8056  0.0000  0.0000
상한0          0.7667  0.7333  0.8469  0.8376  0.8241  0.8056  0.0000  0.0000
상한1          0.6333  0.7000  0.7515  0.8014  0.8241  0.8056  0.1111  0.0000
감점0.02       0.6778  0.7111  0.7788  0.8145  0.8241  0.8056  0.0778  0.0000
후보100 고정     0.6556  0.7222  0.7709  0.8020  0.8241  0.8194  0.0222  0.0000
후보400 고정     0.7556  0.7333  0.8363  0.8329  0.8241  0.8056  0.0000  0.0000
선호가중치0       0.4889  0.4556  0.5751  0.5721  0.5454  0.5435  0.0778  0.0222

## 6. 모드별 관찰

임베딩만 → 조건적용 → 전체 파이프라인 순으로 지표가 어떻게 달라지는지 A/B 각각 본다.

In [8]:
eval_df[eval_df["설정"].isin(MODES)].set_index(["텍스트구성", "설정"]).loc[[("A", m) for m in MODES] + [("B", m) for m in MODES]]

질의수     P@5  nDCG@5     MRR  미판정비율
텍스트구성 설정                                             
A     임베딩만          18  0.3111  0.4040  0.3833    0.0
      조건적용          18  0.5000  0.5885  0.5185    0.0
      조건+재랭킹+중복제어   18  0.7556  0.8363  0.8241    0.0
B     임베딩만          18  0.3111  0.4321  0.4491    0.0
      조건적용          18  0.4444  0.5763  0.5324    0.0
      조건+재랭킹+중복제어   18  0.7333  0.8329  0.8056    0.0

## 7. 질의별 상세 (텍스트 B, 전체 파이프라인)

In [9]:
from dataclasses import replace

per_query = []
for q in QUERIES:
    r = RECS["B"].recommend(q, replace(FULL, top_k=K))
    m = evaluate_result(r, jmap, K)
    per_query.append({"유형": QUERY_KIND[q], **m,
                      "상위5": " / ".join(f"{it['메뉴명']}({jmap.get((q, it['라벨링단위ID']), '?')})" for it in r["추천"])})
per_query_df = pd.DataFrame(per_query).round(3)
per_query_df

,유형,질의,반환수,미판정수,P@5,nDCG@5,RR,적합수,부분적합수,상위5
0,기존,비 오는 날 얼큰한 국물 먹고 싶어,5,0,0.8,0.857,1.0,4,1,해장국 뼈다귀(2) / 국물맵떡(1) / 꽃게 매운탕(2) / 알탕(2) / 복 매운탕(2)
1,기존,맵지 않고 따뜻한 음식,5,0,1.0,1.000,1.0,5,0,꽃맛살쉬림프(2) / 불고기와퍼 버거(2) / 업그레이비타워(2) / 웰빙다이어트야채 피자(2) / 수플레 오믈렛 라이스(2)
2,기존,차갑고 가볍게 먹을 메뉴,5,0,0.0,0.412,0.0,0,4,크랜베리 치킨 치즈 샌드위치(1) / 치킨 샐러드 통밀 샌드위치(1) / 소고기 감자죽(1) / 간편조리세트 매콤제육비빔면(1) / 간편조리세트 차돌박이숙...
3,기존,바삭하고 기름진 음식,5,0,1.0,1.000,1.0,5,0,아빠의제주깜슐랭 치킨(2) / 바삭담백한 후라이드 치킨(2) / 바삭몬테크리스토(2) / 해쉬브라운&치킨 버거(2) / 아 귀한 먹태(2)
4,기존,든든한 밥 한 끼,5,0,0.8,0.913,1.0,4,1,잡탕밥(2) / 육회비빔밥(2) / 짜장밥(2) / 덮밥 닭고기(2) / 간편조리세트 부채살 찹스테이크(1)
5,기존,국물 없는 매운 음식,5,0,1.0,1.000,1.0,5,0,매운 양념 치킨(2) / 쟁반국수(2) / 매운불고기 피자(2) / 직화매운갈비 피자(2) / 막국수(2)
6,기존,상큼하고 시원한 음식,5,0,0.0,0.255,0.0,0,3,씨앗곡물튜나샌드(1) / 쿠키 & 크림 샌디(1) / 반미터 맛있는 피자(0) / 상큼하와이안 피자(1) / 송어 매운탕(0)
7,기존,포만감 있는 저녁밥,5,0,1.0,1.000,1.0,5,0,소고기 덮밥(2) / 덮밥 해물(2) / 하이라이스(2) / 잡탕밥(2) / 자장밥(2)
8,기존,빠르게 먹을 수 있는 간식,5,0,0.4,0.563,0.5,2,3,다이어트싱글 버거(1) / 주먹밥(2) / 채소죽(1) / 땅콩죽(1) / 채소 꼬치구이(2)
9,기존,따뜻한 국이나 찌개,5,0,1.0,1.000,1.0,5,0,두부찌개(2) / 감자 소고기찌개(2) / 섞어찌개(모듬찌개)(2) / 굴 두부찌개(2) / 조기찌개(2)


In [10]:
worst = per_query_df.sort_values(f"nDCG@{K}").head(5)
print("nDCG가 낮은 질의 5개:")
for _, row in worst.iterrows():
    print(f"  {row['질의']!r}: nDCG={row[f'nDCG@{K}']}, P@5={row[f'P@{K}']}, 미판정={row['미판정수']}")
    print(f"     {row['상위5']}")

nDCG가 낮은 질의 5개:
  '상큼하고 시원한 음식': nDCG=0.255, P@5=0.0, 미판정=0
     씨앗곡물튜나샌드(1) / 쿠키 & 크림 샌디(1) / 반미터 맛있는 피자(0) / 상큼하와이안 피자(1) / 송어 매운탕(0)
  '피자 먹고 싶은데 느끼하지 않은 걸로': nDCG=0.409, P@5=0.0, 미판정=0
     땡초참치마요 피자(1) / 꽃맛살쉬림프 피자(1) / 반미터 생각나는 피자(1) / 불고기 피자(1) / 피자 불고기피자(1)
  '차갑고 가볍게 먹을 메뉴': nDCG=0.412, P@5=0.0, 미판정=0
     크랜베리 치킨 치즈 샌드위치(1) / 치킨 샐러드 통밀 샌드위치(1) / 소고기 감자죽(1) / 간편조리세트 매콤제육비빔면(1) / 간편조리세트 차돌박이숙주볶음(0)
  '빠르게 먹을 수 있는 간식': nDCG=0.563, P@5=0.4, 미판정=0
     다이어트싱글 버거(1) / 주먹밥(2) / 채소죽(1) / 땅콩죽(1) / 채소 꼬치구이(2)
  '단짠단짠한 음식': nDCG=0.766, P@5=0.6, 미판정=0
     단짠반반 피자(2) / 단짠콘후라이 피자(2) / 단짠갈릭 치킨(2) / 징거더블다운통다리(0) / 분짜(1)


## 8. 설정 비교 요약

같은 정답 세트에서 텍스트 A/B와 설정을 비교한다. 아래 차이는 모델 추정 정답 기준이며, 승인 후 값이 바뀔 수 있다.

In [11]:
best = eval_df.sort_values(f"nDCG@{K}", ascending=False).head(8)
best[["텍스트구성", "설정", f"P@{K}", f"nDCG@{K}", "MRR", "미판정비율"]]

,텍스트구성,설정,P@5,nDCG@5,MRR,미판정비율
3,A,상한0,0.7667,0.8469,0.8241,0.0
12,B,상한0,0.7333,0.8376,0.8056,0.0
2,A,조건+재랭킹+중복제어,0.7556,0.8363,0.8241,0.0
7,A,후보400 고정,0.7556,0.8363,0.8241,0.0
16,B,후보400 고정,0.7333,0.8329,0.8056,0.0
11,B,조건+재랭킹+중복제어,0.7333,0.8329,0.8056,0.0
14,B,감점0.02,0.7111,0.8145,0.8056,0.0
15,B,후보100 고정,0.7222,0.8020,0.8194,0.0


In [12]:
# A/B 차이: 같은 설정에서 B − A
ab = eval_df.pivot(index="설정", columns="텍스트구성", values=f"nDCG@{K}")
ab["B−A"] = (ab["B"] - ab["A"]).round(4)
ab.loc[list(CONFIGS)]

텍스트구성,A,B,B−A
설정,,,
임베딩만,0.4040,0.4321,0.0281
조건적용,0.5885,0.5763,-0.0122
조건+재랭킹+중복제어,0.8363,0.8329,-0.0034
상한0,0.8469,0.8376,-0.0093
상한1,0.7515,0.8014,0.0499
감점0.02,0.7788,0.8145,0.0357
후보100 고정,0.7709,0.8020,0.0311
후보400 고정,0.8363,0.8329,-0.0034
선호가중치0,0.5751,0.5721,-0.0030


## 9. 파서 처리 결과와 지표

조건이 추출되지 않은 질의(미처리만 있는 경우)와 필수 조건이 있는 질의를 나눠 본다.

In [13]:
def kind_of(q):
    p = parse_query(q)
    if p.hard:
        return "필수 조건 있음"
    if p.soft:
        return "선호만"
    return "조건 없음"

per_query_df["조건유형"] = [kind_of(q) for q in per_query_df["질의"]]
per_query_df.groupby("조건유형")[[f"P@{K}", f"nDCG@{K}", "RR", "미판정수"]].mean().round(3)

,P@5,nDCG@5,RR,미판정수
조건유형,,,,
선호만,0.667,0.778,0.722,0.0
조건 없음,0.600,0.790,1.000,0.0
필수 조건 있음,0.857,0.916,0.857,0.0


## 10. 승인된 판정만으로 계산

사람이 검토해 `검토상태`를 `승인`으로 바꾼 행만 쓴다. 승인이 없으면 실행하지 않는다.

In [14]:
approved = judgment_map(rows, approved_only=True)
approved_queries = sorted({q for q, _ in approved})
if approved:
    print(f"승인 판정 {len(approved)}건, 질의 {len(approved_queries)}개")
    approved_df = pd.DataFrame(evaluate_configs(RECS, approved_queries, CONFIGS, approved, k=K)).round(4)
    display(approved_df.pivot(index="설정", columns="텍스트구성", values=[f"P@{K}", f"nDCG@{K}", "MRR", "미판정비율"]))
else:
    approved_df = None
    print("승인된 판정이 없어 미실행. judgments.csv에서 적합도를 확인하고 검토상태를 '승인'으로 바꾼 뒤 다시 실행한다.")

승인된 판정이 없어 미실행. judgments.csv에서 적합도를 확인하고 검토상태를 '승인'으로 바꾼 뒤 다시 실행한다.


## 11. 결과 저장

In [15]:
eval_df.to_csv(OUT_DIR / "eval_configs.csv", index=False, encoding="utf-8-sig")
per_query_df.to_csv(OUT_DIR / "eval_per_query_B_full.csv", index=False, encoding="utf-8-sig")
run_config = {
    "실행시각": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "k": K, "질의": QUERIES, "제외질의": EXCLUDED,
    "판정": {"파일": str(JUDGMENTS_PATH.relative_to(PROJECT_ROOT)), "행수": len(rows), "검토상태": dict(status),
           "판정출처": dict(Counter(r["판정출처"] for r in rows if r["판정출처"]))},
    "풀설정": {"A,B": list(MODES), "B": list(VARIANTS)},
    "평가설정": {name: asdict(cfg) for name, cfg in CONFIGS.items()},
    "임베딩": {v: rec.embedding_ref for v, rec in RECS.items()},
    "주의": "판정은 모델 추정(검토대기) 기준, 미판정은 0으로 계산, 승인 판정 기준 결과는 approved 절 참조",
    "승인기준결과": approved_df.to_dict("records") if approved_df is not None else None,
}
with open(OUT_DIR / "eval_run_config.json", "w", encoding="utf-8") as f:
    json.dump(run_config, f, ensure_ascii=False, indent=2, default=str)
for p in sorted(OUT_DIR.iterdir()):
    print(f"{p.name:28s} {p.stat().st_size:>9,} bytes")

eval_configs.csv                   882 bytes
eval_per_query_B_full.csv        3,648 bytes
eval_run_config.json             6,572 bytes
judgments.csv                   86,615 bytes


## 12. 요약과 한계

관찰 (모델 추정 정답 기준, 위 셀 출력)
- 임베딩만 → 조건적용 → 전체 파이프라인 순으로 P@5와 nDCG@5가 뚜렷이 올라간다 (6절). 필수 조건 필터와 선호 재랭킹이 정답 기준으로도 도움이 된다.
- 필수 조건이 있는 질의가 선호만 있는 질의보다 지표가 높다 (9절). 선호 재랭킹은 라벨 일치에 의존하므로 조건이 명시될수록 잘 맞는다.
- 텍스트 A와 B의 차이는 설정마다 방향이 다르며 작다 (8절 B−A). 어느 쪽이 낫다고 단정할 수 없다.
- 다양성 제어(상한 1, 감점 0.02)는 nDCG를 약간 낮춘다. nDCG는 다양성을 보상하지 않으므로 예상된 결과이며, 다양성의 가치는 5단계의 반복 수 지표로 따로 봐야 한다. A의 상한1·감점·후보100은 미판정이 섞여 있어 값이 낮게 나온다.
- 지표가 낮은 질의는 "상큼하고 시원한 음식"(상큼은 스키마에 없는 맛), "차갑고 가볍게 먹을 메뉴"(차가움 라벨이 72건뿐), "피자 먹고 싶은데 느끼하지 않은 걸로"(기름짐 라벨로는 담백한 피자를 가르기 어려움)다 (7절). 세 경우 모두 파서나 랭킹보다 데이터·라벨 범위의 한계다.
- 조건이 추출되지 않은 질의("단짠단짠한 음식")는 임베딩 유사도에만 의존한다.

한계
- 정답이 모델 추정이다. 사람이 승인하기 전에는 상대 비교의 참고치다.
- 풀링 편향: 풀에 들어간 설정이 유리하다. 새 설정을 평가하려면 그 설정의 상위 결과를 풀에 넣고 판정을 추가해야 한다 (4절이 자동으로 미판정 행을 덧붙인다).
- 질의 18개, 판정 296건은 작다. 질의를 늘리고 판정을 사람이 검토해야 결론을 낼 수 있다.
- 판정 기준(4절 표)이 주관적이며, "부분 적합"의 범위가 질의마다 다르다.

다음에 할 일
- `judgments.csv`를 검토해 적합도를 고치고 `검토상태`를 `승인`으로 바꾼 뒤 이 노트북을 다시 실행한다 (10절이 승인 기준 결과를 낸다).
- 실제 사용자 입력에서 질의를 모아 정답 세트를 늘린다.
- 승인 정답으로 A/B, 가중치, 상한, 후보 수를 결정한다.